<class 'pandas.core.frame.DataFrame'>
Index: 25863 entries, 2 to 80330
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Seconds    25863 non-null  float64
 1   Country    25863 non-null  object 
 2   Latitude   25863 non-null  float64
 3   Longitude  25863 non-null  float64
dtypes: float64(3), object(1)
memory usage: 1010.3+ KB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        41
           1       0.82      0.22      0.35       250
           2       1.00      1.00      1.00         8
           3       1.00      1.00      1.00       131
           4       0.96      1.00      0.98      4743

    accuracy                           0.96      5173
   macro avg       0.96      0.84      0.87      5173
weighted avg       0.96      0.96      0.95      5173

p label :  [4 4 4 ... 3 4 4]
accuracy:  0.9601778465107288
[1]


d:\python3.13.5\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
d:\python3.13.5\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [1]:
import pandas as pd
import numpy as np

# 1. 加载并清洗数据
ufos = pd.read_csv('./data/ufos.csv')
ufos.head()
# 重构DataFrame，只保留需要的列
ufos = pd.DataFrame({
    'Seconds': ufos['duration (seconds)'], 
    'Country': ufos['country'], 
    'Latitude': ufos['latitude'], 
    'Longitude': ufos['longitude']
})
# 查看唯一国家值（可选）
print("原始国家列表：", ufos.Country.unique())
# 原地删除缺失值
ufos.dropna(inplace=True)
# 筛选Seconds在1-60之间的数据
ufos = ufos[(ufos['Seconds'] >= 1) & (ufos['Seconds'] <= 60)]
ufos.info()



原始国家列表： ['us' nan 'gb' 'ca' 'au' 'de']
<class 'pandas.core.frame.DataFrame'>
Index: 25863 entries, 2 to 80330
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Seconds    25863 non-null  float64
 1   Country    25863 non-null  object 
 2   Latitude   25863 non-null  float64
 3   Longitude  25863 non-null  float64
dtypes: float64(3), object(1)
memory usage: 1010.3+ KB


In [2]:
# 2. 标签编码（保存编码器实例，方便转回国家名称）
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()  # 实例化编码器，复用
ufos['Country'] = le.fit_transform(ufos['Country'])
print("编码后的国家映射：", dict(zip(le.classes_, le.transform(le.classes_))))
ufos.head()



编码后的国家映射： {'au': np.int64(0), 'ca': np.int64(1), 'de': np.int64(2), 'gb': np.int64(3), 'us': np.int64(4)}


,Seconds,Country,Latitude,Longitude
2,20.0,3,53.200000,-2.916667
3,20.0,4,28.978333,-96.645833
14,30.0,4,35.823889,-80.253611
23,60.0,4,45.582778,-122.352222
24,3.0,3,51.783333,-0.783333


In [3]:
# 3. 划分特征和目标变量
from sklearn.model_selection import train_test_split
Selected_features = ['Seconds', 'Latitude', 'Longitude']
X = ufos[Selected_features]
y = ufos['Country']
# 划分训练集/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)



In [4]:
# 4. 训练模型（解决收敛警告）
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
# 增加max_iter=1000，解决收敛警告
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# 输出模型评估结果
print("\n=== 模型评估结果 ===")
print(classification_report(y_test, predictions))
print("预测标签示例: ", predictions[:5])  # 只打印前5个，更整洁
print('准确率: ', round(accuracy_score(y_test, predictions), 4))




=== 模型评估结果 ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        41
           1       0.85      0.47      0.60       250
           2       1.00      1.00      1.00         8
           3       1.00      1.00      1.00       131
           4       0.97      1.00      0.98      4743

    accuracy                           0.97      5173
   macro avg       0.96      0.89      0.92      5173
weighted avg       0.97      0.97      0.97      5173

预测标签示例:  [4 4 4 3 4]
准确率:  0.9702


In [5]:
# 5. 保存并加载模型（用with语句更安全）
import pickle
model_filename = 'ufo-model.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(model, f)

# 加载模型
with open(model_filename, 'rb') as f:
    loaded_model = pickle.load(f)



In [6]:
# 6. 预测（用DataFrame传入，消除特征名警告）
# 方法：构造和训练时格式一致的DataFrame（带列名）
sample_data = pd.DataFrame(
    [[50, 44, -12]],  # 顺序：Seconds, Latitude, Longitude
    columns=Selected_features  # 指定特征名，和训练时一致
)
# 预测
prediction_code = loaded_model.predict(sample_data)[0]  # 取第一个结果
prediction_country = le.inverse_transform([prediction_code])[0]  # 转回国家名称

# 输出最终结果
print("\n=== 预测结果 ===")
print(f"输入参数：Seconds=50, Latitude=44, Longitude=-12")
print(f"预测国家编码：{prediction_code}")
print(f"预测国家名称：{prediction_country}")


=== 预测结果 ===
输入参数：Seconds=50, Latitude=44, Longitude=-12
预测国家编码：3
预测国家名称：gb
